# Barlow Twins for NIR Spectroscopy — Kiwifruit Example

Semi-supervised regression predicting dry matter content (DM) from NIR spectra,
using a small labeled subset plus a large unlabeled pool.

This notebook mirrors `Barlow_for_NIR_example.ipynb` but imports all logic
from the `barlow_twins_nir` package.

## 1. Install & Import

In [ ]:
# Install the package (run once in Colab)
# !pip install -e ..  # from repo root, or:
# !pip install barlow-twins-nir

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from livelossplot import PlotLossesKerasTF

from barlow_twins_nir import (
    load_kiwifruit,
    remove_outliers,
    normalize_features,
    make_paired_views,
    make_labeled_dataset,
    make_semi_supervised_dataset,
    make_validation_dataset,
    BarlowRegressionModel,
    build_supervised_model,
    compute_metrics,
    plot_predictions,
)

## 2. Load & Preprocess

In [ ]:
DATA_PATH = '/data/kiwifruit_dat.csv'  # adjust path as needed
X_LOWER = 'X402'
X_UPPER = 'X1065'
TARGET_COL = 'DM'

# Load and filter
kiwi = load_kiwifruit(DATA_PATH, dm_cutoff=7, min_readings=2)
print(f'Loaded: {len(kiwi)} rows, {kiwi["sample_id"].nunique()} unique samples')

In [ ]:
# Remove spectral outliers via PCA + Mahalanobis distance
kiwi = remove_outliers(kiwi, x_lower=X_LOWER, x_upper=X_UPPER,
                       n_components=20, threshold=1200)
print(f'After outlier removal: {len(kiwi)} rows')

In [ ]:
# Build paired views (view A and view B from different NIR devices)
repeated_kiwi_x, repeated_kiwi_y = make_paired_views(kiwi, X_LOWER, X_UPPER)
print(f'Paired views: {len(repeated_kiwi_x)} rows each')

In [ ]:
# Normalize each view using its own training-set statistics
train_mask_x = repeated_kiwi_x['Dataset'] == 'Training'
train_mask_y = repeated_kiwi_y['Dataset'] == 'Training'

features_x = repeated_kiwi_x.loc[:, X_LOWER:X_UPPER]
features_y = repeated_kiwi_y.loc[:, X_LOWER:X_UPPER]

features_x_norm = normalize_features(features_x, train_mask_x)
features_y_norm = normalize_features(features_y, train_mask_y)

N_FEATURES = features_x_norm.shape[1]
print(f'Feature dimensions: {N_FEATURES}')

## 3. Build Datasets

In [ ]:
# Configuration
NSAMP = 100        # number of labeled training samples
BATCH_SIZE = 4000  # unlabeled batch size
ENC_SIZES = (16,)  # encoder hidden layer sizes
REG_SIZES = [1]    # regression head layer sizes

kiwi_train = kiwi[kiwi['Dataset'] == 'Training']
n_repeats = np.ceil(kiwi_train.shape[0] / BATCH_SIZE)

In [ ]:
# Unlabeled dataset — all training paired views
dataset_enc = tf.data.Dataset.from_tensor_slices((
    features_x_norm.loc[repeated_kiwi_x['Dataset'] == 'Training'].values,
    features_y_norm.loc[repeated_kiwi_x['Dataset'] == 'Training'].values,
))
dataset_enc = dataset_enc.shuffle(buffer_size=train_mask_x.sum()).batch(BATCH_SIZE)

In [ ]:
# Labeled dataset — small subset
dataset_label, tail_ids, features_sub_norm, target_sub = make_labeled_dataset(
    kiwi, nsamp=NSAMP, x_lower=X_LOWER, x_upper=X_UPPER,
    target_col=TARGET_COL, batch_size=BATCH_SIZE,
)
print(f'Labeled samples: {len(features_sub_norm)}')

In [ ]:
# Semi-supervised training dataset
training_dataset = make_semi_supervised_dataset(
    dataset_enc, dataset_label, n_repeats=n_repeats
).prefetch(tf.data.AUTOTUNE)

In [ ]:
# Validation and test datasets
kiwi_sorted = kiwi.sort_values(by=['sample_id'])
dataset_val, dataset_test = make_validation_dataset(
    features_x_norm, features_y_norm,
    repeated_kiwi_x, repeated_kiwi_y,
    target_col=TARGET_COL, batch_size=4000,
)

## 4. Train Barlow Twins Model

In [ ]:
barlow_model = BarlowRegressionModel(
    init_shape=(N_FEATURES,),
    enc_sizes=ENC_SIZES,
    reg_sizes=REG_SIZES,
    loss_weight=(10.5, 0.5, 0.5, 0.),
    barlow_lambda=1 / 15,
    BATCH_SIZE=BATCH_SIZE,
    activation='linear',
    conv1=1.,
    preprocess=3,
    initializer=tf.keras.initializers.LecunNormal(seed=123),
)
barlow_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.005, clipvalue=1.)
)

In [ ]:
callbacks_barlow = [
    PlotLossesKerasTF(),
    EarlyStopping(monitor='loss', min_delta=1e-3, patience=50,
                  restore_best_weights=True),
    ReduceLROnPlateau(patience=25, factor=0.5, min_lr=1e-6,
                      monitor='loss', verbose=0),
    keras.callbacks.ModelCheckpoint(
        filepath='model4a_weights.keras', verbose=1, save_best_only=True
    ),
]

history_barlow = barlow_model.fit(
    training_dataset,
    epochs=1000,
    validation_data=dataset_val,
    callbacks=callbacks_barlow,
)

## 5. Train Supervised Baseline

In [ ]:
# Normalize features for baseline (uses original kiwi, not paired views)
features_all = kiwi.loc[:, X_LOWER:X_UPPER]
train_mask_all = kiwi['Dataset'] == 'Training'
features_norm = normalize_features(features_all, train_mask_all)

In [ ]:
sup_model = build_supervised_model(
    init_shape=(N_FEATURES,),
    enc_sizes=ENC_SIZES,
    reg_sizes=REG_SIZES,
    activation='linear',
    conv1=1.,
    preprocess=3,
    learning_rate=0.005,
    initializer=tf.keras.initializers.glorot_uniform(seed=123),
)

In [ ]:
callbacks_sup = [
    PlotLossesKerasTF(),
    EarlyStopping(monitor='loss', min_delta=1e-3, patience=150,
                  restore_best_weights=True),
    ReduceLROnPlateau(patience=25, factor=0.5, min_lr=1e-6,
                      monitor='loss', verbose=0),
    keras.callbacks.ModelCheckpoint(
        filepath='model_weights.keras', verbose=1, save_best_only=True
    ),
]

history_sup = sup_model.fit(
    features_sub_norm,
    target_sub,
    epochs=1000,
    batch_size=500,
    validation_data=(
        features_norm.loc[kiwi['Dataset'] == 'Validation'].values,
        kiwi.loc[kiwi['Dataset'] == 'Validation', TARGET_COL].values,
    ),
    callbacks=callbacks_sup,
)

## 6. Evaluate & Compare

In [ ]:
# --- Barlow Twins ---
non_train_mask = repeated_kiwi_x['Dataset'] != 'Training'
y_pred_barlow = barlow_model.predict(
    features_x_norm.loc[non_train_mask].values
)[:, 0]
y_true_barlow = repeated_kiwi_x.loc[non_train_mask, TARGET_COL].values

metrics_barlow = compute_metrics(y_true_barlow, y_pred_barlow)
print('Barlow Twins — RMSE: {rmse:.3f}  R²: {r2:.3f}'.format(**metrics_barlow))

In [ ]:
# --- Supervised Baseline ---
non_train_mask_kiwi = kiwi['Dataset'] != 'Training'
y_pred_sup = sup_model.predict(
    features_norm.loc[non_train_mask_kiwi].values
)[:, 0]
y_true_sup = kiwi.loc[non_train_mask_kiwi, TARGET_COL].values

metrics_sup = compute_metrics(y_true_sup, y_pred_sup)
print('Supervised    — RMSE: {rmse:.3f}  R²: {r2:.3f}'.format(**metrics_sup))

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 7))

fig1 = plot_predictions(y_true_barlow, y_pred_barlow,
                        title=f'Barlow Twins (n={NSAMP})',
                        save_path='barlow_predictions.png')
fig2 = plot_predictions(y_true_sup, y_pred_sup,
                        title=f'Supervised Baseline (n={NSAMP})',
                        save_path='supervised_predictions.png')
plt.show()